## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import joblib

sns.set_style("whitegrid")
%matplotlib inline

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 2. Load the Dataset


In [ ]:
def generate_house_data(n_samples=1000, random_state=42):
    rng = np.random.default_rng(random_state)

    area_sqft = rng.normal(1800, 650, n_samples).clip(300, 6000)
    bedrooms = rng.integers(1, 6, n_samples)
    bathrooms = rng.integers(1, 4, n_samples)
    age_years = rng.integers(0, 60, n_samples)
    garage_spaces = rng.integers(0, 3, n_samples)
    distance_to_city_km = rng.exponential(8, n_samples).clip(0.5, 60)
    location_score = rng.uniform(1, 10, n_samples)  # neighborhood desirability

    # Ground-truth linear relationship + noise, so Linear Regression is a sensible model
    price = (
        50_000
        + area_sqft * 120
        + bedrooms * 8_000
        + bathrooms * 6_000
        - age_years * 900
        + garage_spaces * 5_000
        - distance_to_city_km * 1_500
        + location_score * 9_000
        + rng.normal(0, 20_000, n_samples)
    ).clip(30_000, None)

    df = pd.DataFrame({
        "area_sqft": area_sqft.round(0),
        "bedrooms": bedrooms,
        "bathrooms": bathrooms,
        "age_years": age_years,
        "garage_spaces": garage_spaces,
        "distance_to_city_km": distance_to_city_km.round(2),
        "location_score": location_score.round(2),
        "price": price.round(0),
    })
    return df

df = generate_house_data(n_samples=1000, random_state=RANDOM_STATE)
print(df.shape)
df.head()


(1000, 8)


,area_sqft,bedrooms,bathrooms,age_years,garage_spaces,distance_to_city_km,location_score,price
0,1998.0,1,2,5,2,12.56,2.63,319589.0
1,1124.0,4,2,46,1,6.02,4.54,241153.0
2,2288.0,3,1,50,0,0.50,6.82,392801.0
3,2411.0,2,3,56,2,0.75,4.78,358234.0
4,532.0,4,2,19,0,18.77,2.57,134180.0


## 3. Data Preprocessing


In [ ]:
'''
FEATURES = [
    "area_sqft", "bedrooms", "bathrooms", "age_years",
    "garage_spaces", "distance_to_city_km", "location_score",
]'''
FEATURES = [
    "area_sqft","bedrooms","bathrooms"
]
TARGET = "price"

X = df[FEATURES]
y = df[TARGET]


## 5. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")


Train size: 800, Test size: 200


## 6. Train the Linear Regression Model

In [ ]:
model = LinearRegression()
model.fit(X_train_scaled, y_train)


LinearRegression()

## 7. Evaluate the Model

In [ ]:
y_pred = model.predict(X_test_scaled)
r2 = r2_score(y_test, y_pred)
print(f"R^2  : {r2:.4f}")


R^2  : 0.8261


## 9. Save the Model


In [ ]:
import os
joblib.dump(model, "house_price_linear_regression.joblib")

['house_price_linear_regression.joblib']

## 10. Predict on New Data

In [ ]:
predicted_price = model.predict([[700,1,6]])
print(f"Predicted price for the new house: ${predicted_price[0]:,.2f}")

Predicted price for the new house: $53,424,182.06
